# 01 — Exploratory Data Analysis
** (Data Engineer)
**Project:** Hackathon Success & AI Career Readiness Prediction

This notebook explores `data/raw/student_employability.csv` — a **synthetic**
dataset (see `data_dictionary.csv` and the generator script
`src/generate_dataset.py` for how it was built). We look at:

1. Basic structure & data types
2. Missing values
3. Target distributions (`job_ready`, `career_track`)
4. Feature distributions
5. Correlations
6. The central hackathon question: does hackathon participation
   correlate with job readiness, and how does it compare to other signals?

> **Note:** Because this data is synthetic, any patterns found here
> demonstrate *methodology*, not real-world employability claims.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (8, 5)

df = pd.read_csv("../data/raw/student_employability.csv")
df.shape

## 1. Basic structure & data types

In [ ]:
df.info()

In [ ]:
df.describe(include="all").T

## 2. Missing values

In [ ]:
missing = df.isna().sum()
missing = missing[missing > 0].sort_values(ascending=False)
missing_pct = (missing / len(df) * 100).round(2)
pd.DataFrame({"missing_count": missing, "missing_pct": missing_pct})

In [ ]:
plt.figure(figsize=(8, 4))
sns.barplot(x=missing.index, y=missing.values, color="#4C72B0")
plt.title("Missing Values per Column")
plt.ylabel("Number of missing rows")
plt.xticks(rotation=30, ha="right")
plt.tight_layout()
plt.savefig("eda_assets/missing_values.png", dpi=120)
plt.show()

**Observation:** `best_competition_rank` is missing whenever a student
entered zero Kaggle competitions — this is *structural* missingness, not
random, and should be handled with a missing-indicator flag rather than
naive imputation. The other columns (`cgpa`, `certifications`,
`communication_score`, `resume_score`) have a small (~3%) random missing
rate typical of survey data and can be median-imputed.

## 3. Target distributions

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

df["job_ready"].value_counts().sort_index().plot(
    kind="bar", ax=axes[0], color=["#C44E52", "#55A868"]
)
axes[0].set_title("Job Ready Distribution")
axes[0].set_xticklabels(["Not Ready (0)", "Ready (1)"], rotation=0)
axes[0].set_ylabel("Count")

track_labels = ["Data Analyst", "Data Scientist", "ML Engineer",
                "DL Engineer", "GenAI Engineer", "Not Yet Ready"]
counts = df["career_track"].value_counts().sort_index()
axes[1].bar(track_labels, counts.values, color="#4C72B0")
axes[1].set_title("Career Track Distribution")
axes[1].tick_params(axis="x", rotation=30)

plt.tight_layout()
plt.savefig("eda_assets/target_distributions.png", dpi=120)
plt.show()

## 4. Feature distributions

In [ ]:
skill_cols = ["python_score", "sql_score", "statistics_score",
              "ml_score", "dl_score", "genai_score"]

fig, axes = plt.subplots(2, 3, figsize=(15, 8))
for ax, col in zip(axes.flat, skill_cols):
    sns.histplot(df[col], kde=True, ax=ax, color="#4C72B0")
    ax.set_title(col)
plt.tight_layout()
plt.savefig("eda_assets/skill_distributions.png", dpi=120)
plt.show()

In [ ]:
experience_cols = ["hackathons_attended", "hackathons_won",
                    "end_to_end_projects", "deployed_projects",
                    "internship_months", "kaggle_competitions"]

fig, axes = plt.subplots(2, 3, figsize=(15, 8))
for ax, col in zip(axes.flat, experience_cols):
    sns.histplot(df[col], discrete=True, ax=ax, color="#55A868")
    ax.set_title(col)
plt.tight_layout()
plt.savefig("eda_assets/experience_distributions.png", dpi=120)
plt.show()

## 5. Correlations

In [ ]:
numeric_cols = df.select_dtypes(include=[np.number]).drop(
    columns=["job_ready", "career_track"]
).columns

corr = df[numeric_cols].corr()

plt.figure(figsize=(14, 11))
sns.heatmap(corr, cmap="coolwarm", center=0, annot=False, square=True)
plt.title("Feature Correlation Heatmap")
plt.tight_layout()
plt.savefig("eda_assets/correlation_heatmap.png", dpi=120)
plt.show()

In [ ]:
corr_with_target = df[numeric_cols.tolist() + ["job_ready"]].corr()["job_ready"] \
    .drop("job_ready").sort_values(key=abs, ascending=False)
corr_with_target

## 6. The Central Hackathon Question

Does attending more hackathons correlate with job readiness — and how
does that compare to fundamentals, projects, deployment, and internship
experience?

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

for ax, col, title in zip(
    axes,
    ["hackathons_attended", "hackathons_won", "internship_months"],
    ["Hackathons Attended vs Job Ready", "Hackathons Won vs Job Ready",
     "Internship Months vs Job Ready"],
):
    sns.boxplot(x="job_ready", y=col, hue="job_ready", data=df, ax=ax,
                palette=["#C44E52", "#55A868"], legend=False)
    ax.set_title(title)
    ax.set_xticks([0, 1])
    ax.set_xticklabels(["Not Ready", "Ready"])

plt.tight_layout()
plt.savefig("eda_assets/hackathon_vs_readiness.png", dpi=120)
plt.show()

In [ ]:
comparison_features = [
    "hackathons_attended", "hackathons_won", "end_to_end_projects",
    "deployed_projects", "internship_months", "ml_score",
    "ml_interview_score",
]

corr_with_target[comparison_features].sort_values(ascending=False)

**Interpretation guide (fill this in with YOUR actual numbers once you run
the notebook — do not copy conclusions from this template):**

- If `hackathons_attended` shows a much *weaker* correlation with
  `job_ready` than `end_to_end_projects`, `deployed_projects`,
  `internship_months`, or `ml_interview_score`, that supports the
  project's hypothesis that hackathon attendance alone is a weak signal
  compared to demonstrated project/interview evidence.
- If `hackathons_won` correlates more strongly than
  `hackathons_attended`, that suggests *winning* carries more signal than
  mere *participation* — consistent with the guide's framing.
- Because this is a synthetic dataset built with these exact weights
  baked in (see `generate_dataset.py`), these results should be read as a
  demonstration of the analysis method, not as proof about real hiring.
  When you swap in a real dataset, re-run this notebook and let the real
  correlations speak for themselves.

**Next notebook:** `02_preprocessing.ipynb` — cleaning, imputation, and
building the preprocessing pipeline (Vadika).